# 공고돋보기 — 검증 노트북 (eval)
테스트셋 53건(가상, AI로 초안 작성)과 외부 확인셋 4건(공식 자료의 공고 문구 원문)으로 판정 품질을 잰다.

| 지표 | 정의 |
|---|---|
| 탐지율(사기) | 예측 등급이 🔴·🟠·🟡 중 하나인 비율 (엄격 일치 = 허용 등급 안) |
| 오탐률(정상) | 정상 공고가 🔴·🟠·🟡로 나온 비율, 정상 클래스별 |
| 신종 탐지율 | '유형 하나 빼기': 그 유형의 정답 규칙을 지우고 다시 판정했을 때 🔴·🟠·🟡 비율 (AI 추출 재사용, 무비용) |
| 인용 검증 통과율 | AI가 낸 근거 문장 중 마스킹된 원문에서 코드로 확인된 비율 |
| 기준선 | ① AI 없이 rules.yaml 예시 표현 키워드 일치만 ② 같은 모델에 공고만 주고 위험 여부를 한 번 묻기 |

- AI에는 **마스킹된 가상 데이터만** 보낸다. 호출 결과는 `results/` 캐시에 저장해 다시 실행할 때 비용이 들지 않는다.
- 한계: 테스트셋은 규칙을 만든 쪽이 작성해 순환 위험이 있다(외부 확인셋으로 일부 보완). '유형 하나 빼기'는 추출을 재사용하므로, 실제로 규칙이 프롬프트에 없을 때보다 낙관적일 수 있다.

In [1]:
# 환경 준비: Colab 이면 배포용 공개 저장소(app/ 만 담은 저장소)를 받아 오고, 키는 Colab '보안 비밀'에서 읽는다.
import os, sys, json, pathlib, warnings
warnings.filterwarnings("ignore")  # 라이브러리 경고(로컬 경로가 찍힘)를 실행 기록에 남기지 않는다
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    APP_REPO_URL = "https://github.com/<계정>/<배포용 저장소>.git"  # 제출 시 공개되는 저장소 주소로 바꾼다
    if not pathlib.Path("app").exists():
        !git clone -q $APP_REPO_URL app
    !pip install -q -r app/requirements.txt
    from google.colab import userdata
    for k in ["OPENAI_API_KEY", "MODEL_ID", "HF_TOKEN", "HF_DATASET_REPO"]:
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
    APP_DIR = pathlib.Path("app").resolve()
else:
    APP_DIR = pathlib.Path.cwd().resolve().parent  # notebooks/ 의 상위 = app/
sys.path.insert(0, str(APP_DIR))
RESULTS = APP_DIR / "notebooks" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
import pandas as pd
pd.set_option("display.max_colwidth", 80)
from core import config
print("app:", APP_DIR.name, "| model:", config.model_id(), "| OpenAI 키:", "있음" if config.get_secret("OPENAI_API_KEY") else "없음")

app: app | model: gpt-6.2-sol | OpenAI 키: 있음


In [2]:
RUN_AI = True                 # 테스트셋·외부 확인셋 AI 판정 (캐시에 있으면 건너뜀)
RUN_GENERAL_BASELINE = True   # 기준선 ② (같은 모델에 위험 여부만 묻기)
SAVE_SYNTHETIC_CASES = True   # '유형 하나 빼기'의 신종 의심 사례를 '평가용 가상 사례'로 HF 에 저장(학습 고리 시험용)

from core import evaluation as ev
from core.extract import OpenAIExtractor, make_client
from core.rules_engine import load_rules
RB = load_rules()
items = ev.load_items()
ext_items = ev.load_items(ev.EXTERNAL_SET_PATH)
groups = {**ev.load_groups(), **ev.load_groups(ev.EXTERNAL_SET_PATH)}
print(f"테스트셋 {len(items)}건 (사기 {sum(i['class']=='scam' for i in items)} / 정상 {sum(i['class']=='normal' for i in items)}), 외부 확인셋 {len(ext_items)}건, 규칙 v{RB.version}")

테스트셋 53건 (사기 33 / 정상 20), 외부 확인셋 4건, 규칙 v2.1.4


## 1. AI 판정 (마스킹 → 구조화 추출 → 규칙 엔진)

In [3]:
client = make_client() if RUN_AI else None
factory = lambda: OpenAIExtractor(client=client, rulebook=RB)
cache = ev.extract_all(items, factory, RESULTS / "eval_cache_testset.json") if RUN_AI else json.loads((RESULTS / "eval_cache_testset.json").read_text(encoding="utf-8"))
ext_cache = ev.extract_all(ext_items, factory, RESULTS / "eval_cache_external.json") if RUN_AI else json.loads((RESULTS / "eval_cache_external.json").read_text(encoding="utf-8"))
res = ev.judge_items(items, cache, RB)
ext_res = ev.judge_items(ext_items, ext_cache, RB)
s = ev.summarize(res, groups)
display(pd.DataFrame(s["groups"]))
display(pd.DataFrame([s["overall"]]))

,그룹,이름,구분,N,탐지율(🔴🟠🟡),엄격 일치,예측 등급,오탐률(🔴🟠🟡)
0,G1,구매대행·리뷰·팀미션,사기,3,1.0,1.00,"red, red, red",NaN
1,G2,유명 브랜드 사칭 리뷰 체험단,사기,3,1.0,1.00,"orange, orange, red",NaN
2,G3,개인정보 탈취,사기,3,1.0,1.00,"red, red, red",NaN
3,G4,계정 대여·휴대폰 개통,사기,3,1.0,1.00,"red, red, red",NaN
4,G5,마약 등 물품 운반,사기,3,1.0,1.00,"red, red, red",NaN
5,G6,해외 취업,사기,3,1.0,1.00,"red, red, red",NaN
6,G7,스팸문자 전송·지인 모집 보상,사기,3,1.0,1.00,"red, red, red",NaN
7,G12,송금·인출 대행,사기,3,1.0,1.00,"red, red, red",NaN
8,G13,채용 가장 판매원·다단계·수강생 모집,사기,3,1.0,1.00,"yellow, yellow, red",NaN
9,G14,성매매 유인 의심,사기,3,1.0,1.00,"yellow, yellow, yellow",NaN


,사기 N,탐지율,사기 엄격 일치,정상 N,오탐률,정상 엄격 일치,인용 검증 통과율,인용 수,건당 평균 비용($),건당 평균 시간(초),오류 수
0,33,1.0,1.0,20,0.05,0.95,1.0,118,0.0142,10.9,0


## 2. 외부 확인셋 (공식 자료 공고 문구 원문)

In [4]:
se = ev.summarize(ext_res, groups)
display(pd.DataFrame(se["groups"]))
display(pd.DataFrame([{"id": r.id, "기대": r.expected_grade, "예측": r.predicted, "걸린 규칙": ",".join(r.matched_ids)} for r in ext_res]))

,그룹,이름,구분,N,탐지율(🔴🟠🟡),엄격 일치,예측 등급
0,X1,해외 취업사기 문구 (E60),사기,1,1.0,1.0,red
1,X2,스팸문자 전송 알바 문구 (E63),사기,1,1.0,1.0,red
2,X3,성매매 의심 구인광고 예시 (E66),사기,1,1.0,1.0,yellow
3,X4,리뷰 체험단 사기 첫 메시지 (E59),사기,1,0.0,1.0,white


,id,기대,예측,걸린 규칙
0,X1-1,red,red,"H7,AX_F,S1,S3,S4"
1,X2-1,red,red,"H6,S3"
2,X3-1,yellow,yellow,"M5,AX_F,S1"
3,X4-1,white,white,S3


## 3. 기준선 비교

In [5]:
kw = ev.keyword_baseline(items); kw_ext = ev.keyword_baseline(ext_items)  # 업체명 추출이 필요한 M2 는 제외(공정 비교)
gen = gen_ext = None
if RUN_GENERAL_BASELINE:
    gc = client or make_client()
    gen = ev.general_prompt_baseline(items, gc, config.model_id(), RESULTS / "eval_cache_general_testset.json")
    gen_ext = ev.general_prompt_baseline(ext_items, gc, config.model_id(), RESULTS / "eval_cache_general_external.json")
def row(name, r, r_ext):
    o = ev.summarize(r)["overall"]; oe = ev.summarize(r_ext)["overall"] if r_ext else {}
    return {"방식": name, "탐지율": o["탐지율"], "사기 엄격 일치": o["사기 엄격 일치"], "오탐률": o["오탐률"],
            "외부 확인셋 탐지율": oe.get("탐지율"), "건당 비용($)": o["건당 평균 비용($)"]}
compare = [row("공고돋보기 (AI 추출 + 규칙 엔진)", res, ext_res), row("기준선① 키워드만 (AI 없음)", kw, kw_ext)]
if gen:
    compare.append(row("기준선② 범용 프롬프트 한 번", gen, gen_ext))
display(pd.DataFrame(compare))
display(pd.DataFrame(ev.summarize(kw, groups)["groups"])[["그룹", "구분", "N", "탐지율(🔴🟠🟡)", "오탐률(🔴🟠🟡)", "예측 등급"]])
if gen:
    display(pd.DataFrame(ev.summarize(gen, groups)["groups"])[["그룹", "구분", "N", "탐지율(🔴🟠🟡)", "오탐률(🔴🟠🟡)", "예측 등급"]])

,방식,탐지율,사기 엄격 일치,오탐률,외부 확인셋 탐지율,건당 비용($)
0,공고돋보기 (AI 추출 + 규칙 엔진),1.000,1.000,0.05,0.75,0.0142
1,기준선① 키워드만 (AI 없음),0.364,0.333,0.10,0.75,NaN
2,기준선② 범용 프롬프트 한 번,1.000,0.970,0.05,0.75,0.0011


,그룹,구분,N,탐지율(🔴🟠🟡),오탐률(🔴🟠🟡),예측 등급
0,G1,사기,3,0.000,NaN,"white, white, white"
1,G2,사기,3,0.333,NaN,"white, white, red"
2,G3,사기,3,0.000,NaN,"white, white, white"
3,G4,사기,3,0.000,NaN,"white, white, white"
4,G5,사기,3,0.333,NaN,"red, white, white"
5,G6,사기,3,1.000,NaN,"red, red, yellow"
6,G7,사기,3,0.000,NaN,"white, white, white"
7,G12,사기,3,1.000,NaN,"red, red, red"
8,G13,사기,3,0.667,NaN,"yellow, white, yellow"
9,G14,사기,3,0.667,NaN,"yellow, red, white"


,그룹,구분,N,탐지율(🔴🟠🟡),오탐률(🔴🟠🟡),예측 등급
0,G1,사기,3,1.0,NaN,"red, red, red"
1,G2,사기,3,1.0,NaN,"red, red, red"
2,G3,사기,3,1.0,NaN,"red, red, red"
3,G4,사기,3,1.0,NaN,"red, red, red"
4,G5,사기,3,1.0,NaN,"red, red, red"
5,G6,사기,3,1.0,NaN,"red, red, red"
6,G7,사기,3,1.0,NaN,"red, yellow, red"
7,G12,사기,3,1.0,NaN,"red, red, red"
8,G13,사기,3,1.0,NaN,"yellow, yellow, yellow"
9,G14,사기,3,1.0,NaN,"red, red, red"


## 4. 신종 탐지율 ('유형 하나 빼기', 무비용)

In [6]:
loo, removed = ev.leave_one_out(items, cache)
rows = []
for g, rm in removed.items():
    rs = [r for r in loo if r.group == g]
    rows.append({"유형": g, "이름": groups[g]["name"], "뺀 규칙": ",".join(rm), "N": len(rs),
                 "탐지(🔴🟠🟡)": sum(r.flagged for r in rs), "그중 신종 의심 표시": sum(r.novel for r in rs),
                 "예측 등급": ", ".join(str(r.predicted) for r in rs)})
loo_df = pd.DataFrame(rows); display(loo_df)
loo_scam = [r for r in loo]
novel_rate = round(sum(r.flagged for r in loo_scam) / len(loo_scam), 3)
print("신종 탐지율(유형 하나 빼기):", novel_rate, f"({sum(r.flagged for r in loo_scam)}/{len(loo_scam)})")

,유형,이름,뺀 규칙,N,탐지(🔴🟠🟡),그중 신종 의심 표시,예측 등급
0,G1,구매대행·리뷰·팀미션,H3,3,3,3,"red, red, red"
1,G2,유명 브랜드 사칭 리뷰 체험단,"H3,I1,M1",3,1,1,"white, white, red"
2,G3,개인정보 탈취,"H1,H4,M1",3,3,3,"red, red, red"
3,G4,계정 대여·휴대폰 개통,H5,3,3,3,"red, red, red"
4,G5,마약 등 물품 운반,H2,3,3,3,"red, red, red"
5,G6,해외 취업,H7,3,3,2,"yellow, red, red"
6,G7,스팸문자 전송·지인 모집 보상,"H5,H6",3,2,2,"red, white, red"
7,G12,송금·인출 대행,"H1,H2",3,3,3,"red, red, red"
8,G13,채용 가장 판매원·다단계·수강생 모집,"H3,M3",3,1,0,"green, yellow, white"
9,G14,성매매 유인 의심,M5,3,3,0,"yellow, yellow, yellow"


신종 탐지율(유형 하나 빼기): 0.848 (28/33)


## 5. 비용·결과 저장

In [7]:
costs = [r.cost for r in res if r.cost is not None]
cached = [ (r.usage or {}).get("cached_input_tokens") or 0 for r in res if r.usage]
print(f"테스트셋 판정 1건(1회차 텍스트) 평균 ${sum(costs)/len(costs):.4f} (N={len(costs)}), 평균 캐시 입력 {sum(cached)/len(cached):,.0f} 토큰")
summary = {
    "run_at": pd.Timestamp.now(tz="Asia/Seoul").isoformat(timespec="seconds"), "model": config.model_id(), "rules_version": RB.version,
    "testset": s["overall"], "external": se["overall"], "compare": compare, "novel_detection_rate": novel_rate,
    "leave_one_out": rows, "groups": s["groups"],
}
(RESULTS / "eval_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
(RESULTS / "eval_items.json").write_text(json.dumps(ev.results_to_records(res + ext_res), ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("저장:", (RESULTS / "eval_summary.json").name, (RESULTS / "eval_items.json").name)

테스트셋 판정 1건(1회차 텍스트) 평균 $0.0142 (N=53), 평균 캐시 입력 3,296 토큰
저장: eval_summary.json eval_items.json


## 6. 학습 고리 시험 사례 저장 (평가용 가상 사례)

In [8]:
from core import storage, review
novel = ev.novel_suspects_from_loo(items, cache)
print("'유형 하나 빼기'에서 신종 의심으로 판정된 항목:", [it["id"] for it, _, _ in novel])
store = storage.get_store()
marker = RESULTS / "synthetic_cases_saved.json"  # 다시 실행해도 같은 사례를 또 올리지 않게
saved = []
if marker.exists():
    print("평가용 가상 사례는 이미 저장됨:", json.loads(marker.read_text(encoding="utf-8")))
elif SAVE_SYNTHETIC_CASES and store is not None and isinstance(store, storage.HFDatasetStore):
    for it, rec, v in novel:
        case = review.case_from_eval(it, rec, v)
        saved.append(store.save(case))
    marker.write_text(json.dumps({"count": len(saved), "testset_ids": [it["id"] for it, _, _ in novel]}, ensure_ascii=False), encoding="utf-8")
    print("저장한 평가용 가상 사례:", len(saved))

'유형 하나 빼기'에서 신종 의심으로 판정된 항목: ['G1-1', 'G1-2', 'G1-3', 'G2-3', 'G3-1', 'G3-2', 'G3-3', 'G4-1', 'G4-2', 'G4-3', 'G5-1', 'G5-2', 'G5-3', 'G6-2', 'G6-3', 'G7-1', 'G7-3', 'G12-1', 'G12-2', 'G12-3', 'G15-1', 'G15-2', 'G15-3']
평가용 가상 사례는 이미 저장됨: {'count': 22, 'testset_ids': ['G1-1', 'G1-2', 'G1-3', 'G2-3', 'G3-1', 'G3-2', 'G4-1', 'G4-2', 'G4-3', 'G5-1', 'G5-2', 'G5-3', 'G6-2', 'G6-3', 'G7-3', 'G12-1', 'G12-2', 'G12-3', 'G14-1', 'G15-1', 'G15-2', 'G15-3'], 'saved_at': '2026-11-23T04:39+09:00'}
